# Figure 4: Drag coefficient and atmospheric stability

(a) Drag coefficient against wind speed for the ERA5 variable Charnock parameter and fixed Charnock values; (b) difference between neutral and actual 10 m wind speed, separating stable and unstable conditions.

From Özkan et al. (2025), *Sensitivity of global storm surge modelling to sea surface drag*, Ocean Dynamics, https://doi.org/10.1007/s10236-025-01713-3

The processed input files are not included in this repository; this notebook shows how the figure was made. Experiment names: in the code, `cc` = FIX, `cv` = NST, `ntr`/`uv10n` = CAD and `rho`/`uv10rho` = IFS (see the main README).

In [ ]:
# Folder with the processed input files (not included in this repository)
DATA_DIR = "path/to/paper1_data/"

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import proplot as pplt
from scipy.stats import gaussian_kde
import os

In [ ]:
# File paths
chnk_wind_dir = DATA_DIR + "data/1-ERA5/input_for_gtsm/"
tau_dir = DATA_DIR + "data/1-ERA5/processed/TAU/2-CV_TAU/"
cd_dir = DATA_DIR + "data/1-ERA5/processed/TAU/5-Regional_10Y/"
stability_dir = DATA_DIR + "data/1-ERA5/input_for_gtsm/stability_plot_uv10n_uv10/"

In [ ]:
# Load datasets for wind speed vs drag coefficient
storm = "sandy"
lat1, lon1 = 39.460318, -72.692946

ds_wind = xr.open_dataset(chnk_wind_dir + f"charnock_uv10/ERA5_C_UV10_{storm}.nc").sel(latitude=lat1, longitude=lon1, method="nearest")
ds_cd18 = xr.open_dataset(cd_dir + f"merged_CC_CD_b0.018_2006_2015_{storm}.nc").sel(latitude=lat1, longitude=lon1, method="nearest")["cd"] * 1000
ds_cd41 = xr.open_dataset(cd_dir + f"merged_CC_CD_b0.041_2006_2015_{storm}.nc").sel(latitude=lat1, longitude=lon1, method="nearest")["cd"] * 1000
ds_cd60 = xr.open_dataset(cd_dir + f"merged_CC_CD_b0.06_2006_2015_{storm}.nc").sel(latitude=lat1, longitude=lon1, method="nearest")["cd"] * 1000
ds_cd80 = xr.open_dataset(cd_dir + f"merged_CC_CD_b0.08_2006_2015_{storm}.nc").sel(latitude=lat1, longitude=lon1, method="nearest")["cd"] * 1000
ds_cdcv = xr.open_dataset(cd_dir + f"merged_CV_CD_TAU_2006_2015_{storm}.nc").rename({"lat":"latitude","lon":"longitude"})
ds_cdcv = ds_cdcv.sel(latitude=lat1, longitude=lon1, method="nearest")["cd"] * 1000

In [ ]:
# Calculate wind speed
wind_speed = np.sqrt(ds_wind["u10"]**2 + ds_wind["v10"]**2)

# Create DataFrames
df_ws_cd18 = pd.DataFrame({'Wind speed (m/s)': wind_speed.values.flatten(), 'Drag Coefficient': ds_cd18.values.flatten()})
df_ws_cd41 = pd.DataFrame({'Wind speed (m/s)': wind_speed.values.flatten(), 'Drag Coefficient': ds_cd41.values.flatten()})
df_ws_cd60 = pd.DataFrame({'Wind speed (m/s)': wind_speed.values.flatten(), 'Drag Coefficient': ds_cd60.values.flatten()})
df_ws_cd80 = pd.DataFrame({'Wind speed (m/s)': wind_speed.values.flatten(), 'Drag Coefficient': ds_cd80.values.flatten()})
df_ws_cdcv = pd.DataFrame({'Wind speed (m/s)': wind_speed.values.flatten(), 'Drag Coefficient': ds_cdcv.values.flatten()})

In [ ]:
# Load datasets for U10 vs U10N - U10
ds_ws10 = xr.open_dataset(stability_dir + f"ERA5_WS10_{storm}_2006_2015.nc").sel(latitude=lat1, longitude=lon1, method="nearest")
ds_delta_ws10 = xr.open_dataset(stability_dir + f"ERA5_WS10N_minus_WS10_{storm}_2006_2015.nc").sel(latitude=lat1, longitude=lon1, method="nearest")

df_ws10_delta_ws10 = pd.DataFrame({'Wind speed (m/s)': ds_ws10["U10"].values.flatten(), 
                                   'Delta Wind speed (m/s)': ds_delta_ws10["delta_U10"].values.flatten()})

In [ ]:
# Extract data for plotting
wind_speed = df_ws_cdcv['Wind speed (m/s)'].values
drag_coef = df_ws_cdcv['Drag Coefficient'].values
delta_wind_speed = df_ws10_delta_ws10['Delta Wind speed (m/s)'].values
wind_speed_stability = df_ws10_delta_ws10['Wind speed (m/s)'].values

In [ ]:
# KDE calculations
z_drag = gaussian_kde(np.vstack([wind_speed, drag_coef]))(np.vstack([wind_speed, drag_coef]))
z_stability = gaussian_kde(np.vstack([wind_speed_stability, delta_wind_speed]))(np.vstack([wind_speed_stability, delta_wind_speed]))

In [ ]:
# Horizontal plots
fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)

# Plot 1: Wind speed vs Drag Coefficient
scatter1 = axes[0].scatter(wind_speed, drag_coef, c=z_drag, cmap='sunset_r', alpha=0.7, 
                           label='$α_{Chvar}$ (Density)')
axes[0].scatter(df_ws_cd18['Wind speed (m/s)'], df_ws_cd18['Drag Coefficient'], 
                label='$α_{Ch}$=0.018', alpha=0.7, s=15, color='azure')
axes[0].scatter(df_ws_cd41['Wind speed (m/s)'], df_ws_cd41['Drag Coefficient'], 
                label='$α_{Ch}$=0.041', alpha=0.7, s=15, color='medium pink')
axes[0].scatter(df_ws_cd60['Wind speed (m/s)'], df_ws_cd60['Drag Coefficient'], 
                label='$α_{Ch}$=0.060', alpha=0.7, s=15, color='avocado')
axes[0].scatter(df_ws_cd80['Wind speed (m/s)'], df_ws_cd80['Drag Coefficient'], 
                label='$α_{Ch}$=0.080', alpha=0.7, s=15, color='salmon')

# Add Q-Q plot
wind_speed_sorted = np.sort(wind_speed)
drag_sorted = np.sort(drag_coef)
axes[0].plot(wind_speed_sorted, drag_sorted, c='black', linestyle='--', alpha=0.95, #c='bright red',
             lw=2.65, label='$α_{Chvar}$ (Q-Q)')  # Increased transparency

axes[0].set_xlim(0, 27)
axes[0].set_ylim(0, 4.5)
axes[0].set_xlabel('$U_{10}$ (m/s)', fontsize=16)
axes[0].set_ylabel('$C_D$ ($×10^{-3}$)', fontsize=16)
axes[0].legend(markerscale=3, scatterpoints=1, loc='lower right', fontsize=14)
axes[0].tick_params(axis='both', which='major', labelsize=10)

# Add "a" bounding box
axes[0].text(0.02, 0.98, 'a', transform=axes[0].transAxes, fontsize=14, fontweight='bold', 
             bbox=dict(facecolor='white', edgecolor='black'), ha='left', va='top')

# Colorbar for Plot 1
cbar1 = fig.colorbar(scatter1, ax=axes[0], location='bottom', label='Density', pad=0.03, shrink=0.95)
cbar1.ax.tick_params(labelsize=12)  # Increase tick label size
cbar1.set_label('Density', fontsize=14)  # Increase colorbar label size

# Plot 1: Stability (U10 vs U10N - U10)
axes[1].axhspan(0, 1, color='pale salmon', alpha=0.3, zorder=0, label='Unstable')
axes[1].axhspan(-1.7, 0, color='teal', alpha=0.2, zorder=0, label='Stable')
scatter2 = axes[1].scatter(wind_speed_stability, delta_wind_speed, c=z_stability, 
                           cmap='boreal_r', alpha=0.7, zorder=2)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1, zorder=1, alpha=0.5) 
axes[1].set_xlim(-0.5, 27)
axes[1].set_ylim(-1.7, 1)
axes[1].set_xlabel('$U_{10}$ (m/s)', fontsize=16)
axes[1].set_ylabel('$ΔU_{10}$ (m/s)', fontsize=16)
axes[1].legend(loc='lower right', fontsize=14)
axes[1].tick_params(axis='both', which='major', labelsize=10)

# Add "b" bounding box
axes[1].text(0.02, 0.98, 'b', transform=axes[1].transAxes, fontsize=14, fontweight='bold', 
             bbox=dict(facecolor='white', edgecolor='black'), ha='left', va='top')

# Colorbar for Plot 2
cbar2 = fig.colorbar(scatter2, ax=axes[1], location='bottom',
                     label='Density', pad=0.03, shrink=0.95)
cbar2.ax.tick_params(labelsize=12)  # Increase tick label size
cbar2.set_label('Density', fontsize=14)  # Increase colorbar label size


# Save the plot as PDF and PNG, cropping the bounding box
# Define the target directory
save_dir = 'final_plots'
os.makedirs(save_dir, exist_ok=True)  # Create the directory if it doesn't exist

# Save the plot as PDF and PNG in the specified directory
fig.savefig(os.path.join(save_dir, 'scatter_charnock_stability_horizontal_updated.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(save_dir, 'scatter_charnock_stability_horizontal_updated.png'), dpi=600, bbox_inches='tight')

# Show the horizontal plot
plt.show()